# CSIRO Competition Solution Notebook

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID1717006117.jpg
/kaggle/input/csiro-biomass/train/ID1638922597.jpg
/kaggle/input/csiro-biomass/train/ID475010202.jpg
/kaggle/input/csiro-biomass/train/ID1857489997.jpg
/kaggle/input/csiro-biomass/train/ID684383343.jpg
/kaggle/input/csiro-biomass/train/ID605134229.jpg
/kaggle/input/csiro-biomass/train/ID1463690813.jpg
/kaggle/input/csiro-biomass/train/ID1403078396.jpg
/kaggle/input/csiro-biomass/train/ID1997244125.jpg
/kaggle/input/csiro-biomass/train/ID545360459.jpg
/kaggle/input/csiro-biomass/train/ID1783499590.jpg
/kaggle/input/csiro-biomass/train/ID157479394.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID839432753.jpg
/kaggle/input/csiro-biomass/train/ID2030696575.jpg
/kaggle/input/csiro-biomass/train/ID710341728.jpg
/kaggle/input/cs

In [2]:
import shutil
import os

# Copy entire dataset folder
input_folder = "/kaggle/input/csiro-biomass"
output_folder = "/kaggle/working/csiro-biomass"

# Check if input folder exists before copying
if not os.path.exists(output_folder):
    # Copy entire directory
    shutil.copytree(input_folder, output_folder)
    
    print(f"✓ Folder copied to: {output_folder}")
    
    # Update paths
    dataset_path = "/kaggle/working/csiro-biomass/train.csv"
    print(f"dataset_path = '{dataset_path}'")
    
    # List copied files
    print(f"\nCopied files:")
    for item in os.listdir(output_folder):
        item_path = os.path.join(output_folder, item)
        if os.path.isfile(item_path):
            size = os.path.getsize(item_path) / (1024 * 1024)
            print(f"  {item}: {size:.2f} MB")
        else:
            num_files = len(os.listdir(item_path))
            print(f"  {item}/: {num_files} files")
else:
    print("Output folder already exists. Skipping copy.")
    dataset_path = "/kaggle/working/csiro-biomass/train.csv"

Output folder already exists. Skipping copy.


In [3]:
import torch

torch.cuda.is_available()

True

# Data Prep

## Data Augmentation & Transform

In [4]:
# Data Transform

from torchvision.transforms import v2
import torch

# to_tensor = v2.ToTensor()
# img_tensor = to_tensor(img)

dtype = torch.float32
img_size = (224, 224)
image_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype, scale=True),    
    v2.Resize(img_size),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(5, interpolation=v2.InterpolationMode.BILINEAR),
    v2.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.05,
    ),
    # v2.RandomAdjustSharps
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype, scale=True),
    v2.Resize(img_size),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def numeric_transform(X, X_max, X_min) -> torch.Tensor:
    X_normalized = (X - X_min) / (X_max - X_min)
    return X_normalized

def target_transform(targets) -> torch.Tensor:
    return torch.log1p(targets)

def target_untransform(targets) -> torch.Tensor:
    return torch.expm1(targets)

def categorical_transform(row) -> torch.Tensor:
    return row

## Train Set

In [5]:

from torch.utils.data import Dataset
from torchvision.io import decode_image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pandas as pd

class Image2BioMassTrainValDataset(Dataset):
    
    def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None, target_transform=None):
        
        self.df = self.process_df(dataset_path)
        self.dataset_path = dataset_path
        self.img_transform = img_transform
        self.target_transform = target_transform
        self.numeric_transform = numeric_transform
        self.categorical_transform = categorical_transform
        self.targets = self.df.loc[:, ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g"]]


    def process_df(self, dataset_path):
        self.le_date = LabelEncoder()
        self.le_state = LabelEncoder()
        self.le_species = LabelEncoder()

        df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
        df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
        df = df.pivot_table(
        index=['base_sample_id', 'image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
        columns='target_name',
        values='target'
        ).reset_index()
        df["Sampling_Date"] = self.le_date.fit_transform(df["Sampling_Date"])
        df["State"] = self.le_state.fit_transform(df["State"])
        df["Species"] = self.le_species.fit_transform(df["Species"])
        # display(df)
        return df

    def __len__(self):
        return len(self.df)

    def get_cat_features(self):
        return ["Sampling_Date", "State", "Species"]
    
    def get_cat_vocab_sizes(self):
        results = []

        for i in self.get_cat_features():
            results.append(len(self.df[i].unique()))
        return results

    def __getitem__(self, idx):
        # B = batch_size
        # display(self.df)
        img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
        image = decode_image(img_path)
        # display(self.df)
        numeric_features = torch.tensor([
            self.df.loc[idx, "Pre_GSHH_NDVI"],
            self.df.loc[idx, "Height_Ave_cm"],
        ], dtype=torch.float32)

        categorical_features = torch.tensor([
            self.df.loc[idx, "Sampling_Date"],
            self.df.loc[idx, "State"],
            self.df.loc[idx, "Species"],
        ], dtype=torch.long)
        

        if self.img_transform:
            image = self.img_transform(image)
            
        if self.numeric_transform:
            # numeric_features[0] = self.numeric_transform(
            #     numeric_features[0],
            #     self.df.loc[:, "Pre_GSHH_NDVI"].max(), 
            #     self.df.loc[:, "Pre_GSHH_NDVI"].min()
            # )
            numeric_features[1] = self.numeric_transform(
                numeric_features[1], 
                self.df.loc[:, "Height_Ave_cm"].max(), 
                self.df.loc[:, "Height_Ave_cm"].min()
            )
            # print(numeric_features)
        combined_features = torch.cat([categorical_features.float(), numeric_features], dim=0)
        # print(combined_features)
        targets = torch.Tensor(self.targets.iloc[idx].values)
        if self.target_transform:
            targets = self.target_transform(targets)
        return image, combined_features, targets

## Test Set

In [6]:

# from torch.utils.data import Dataset
# from torchvision.io import decode_image
# from sklearn.preprocessing import LabelEncoder
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader
# import pandas as pd

# class Image2BioMassTestFromTrainDataset(Dataset):
    
#     def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None):
        
#         self.df = self.process_df(dataset_path)
#         self.dataset_path = dataset_path
#         self.img_transform = img_transform
#         self.numeric_transform = numeric_transform
#         self.categorical_transform = categorical_transform

#     def process_df(self, dataset_path):
#         self.le_date = LabelEncoder()
#         self.le_state = LabelEncoder()
#         self.le_species = LabelEncoder()

#         df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
#         df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
#         df = (
#             df.assign(_val="")
#               .pivot(index=['base_sample_id', "image_path"],
#                      columns='target_name',
#                      values='_val')
#               .reset_index()
#         )

#         return df

#     def __len__(self):
#         return len(self.df)

#     def get_cat_features(self):
#         return ["Sampling_Date", "State", "Species"]
    
#     def get_cat_vocab_sizes(self):
#         results = []

#         for i in self.get_cat_features():
#             results.append(len(self.df[i].unique()))
#         return results

#     def __getitem__(self, idx):

#         img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
#         image = decode_image(img_path)

#         # Use val_transform for test data (no augmentation)
#         if self.img_transform:
#             image = self.img_transform(image)
#         else:
#             # Fallback basic transform if no transform provided
#             transform = v2.Compose([
#                 v2.ToImage(),
#                 v2.ToDtype(dtype, scale=True),
#                 v2.Resize((518, 518)),
#                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ])
#             image = transform(image)

#         combined_features = torch.zeros(5, dtype=torch.float32)
#         sample_id = self.df.loc[idx, 'base_sample_id']
#         return image, combined_features, sample_id

In [7]:

from torch.utils.data import Dataset
from torchvision.io import decode_image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pandas as pd

class Image2BioMassTestDataset(Dataset):
    
    def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None):
        
        self.df = self.process_df(dataset_path)
        self.dataset_path = dataset_path
        self.img_transform = img_transform
        self.numeric_transform = numeric_transform
        self.categorical_transform = categorical_transform

    def process_df(self, dataset_path):
        self.le_date = LabelEncoder()
        self.le_state = LabelEncoder()
        self.le_species = LabelEncoder()

        df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
        df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
        df = (
            df.assign(_val="")
              .pivot(index=['base_sample_id', "image_path"],
                     columns='target_name',
                     values='_val')
              .reset_index()
        )

        return df

    def __len__(self):
        return len(self.df)

    def get_cat_features(self):
        return ["Sampling_Date", "State", "Species"]
    
    def get_cat_vocab_sizes(self):
        results = []

        for i in self.get_cat_features():
            results.append(len(self.df[i].unique()))
        return results

    def __getitem__(self, idx):

        img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
        image = decode_image(img_path)

        # Use val_transform for test data (no augmentation)
        if self.img_transform:
            image = self.img_transform(image)
        else:
            # Fallback basic transform if no transform provided
            transform = v2.Compose([
                v2.ToImage(),
                v2.ToDtype(dtype, scale=True),
                v2.Resize((518, 518)),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
            image = transform(image)

        combined_features = torch.zeros(5, dtype=torch.float32)
        sample_id = self.df.loc[idx, 'base_sample_id']
        return image, combined_features, sample_id

In [8]:
test_dataset = Image2BioMassTestDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=val_transform,  # Use val_transform (no augmentation, proper size)
    
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)
next(iter(test_dataloader))[2]

('ID1001187975',)

## Train Split

In [9]:
import torch, random, numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

g = torch.Generator()
g.manual_seed(42)

In [10]:

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

# Create base dataset to get indices
base_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=None,  # no transform yet
    numeric_transform=numeric_transform,
    target_transform=target_transform
)

# Split indices
seed = 42
train_indices, val_indices = train_test_split(
    range(len(base_dataset)), 
    train_size=0.8, 
    shuffle=True, 
    random_state=seed
)

# Create training dataset WITH augmentation
train_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=image_transform,  # WITH augmentation
    numeric_transform=numeric_transform,
    target_transform=target_transform
)
train_dataset = Subset(train_dataset, train_indices)

val_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=val_transform,  # WITHOUT augmentation
    numeric_transform=numeric_transform,
    target_transform=target_transform
)
val_dataset = Subset(val_dataset, val_indices)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, generator=g)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Model

In [11]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.models import resnet152, ResNet152_Weights
from torchvision.models import resnet50, ResNet50_Weights
from transformers import Dinov2Model


class BackBone(nn.Module):

    def __init__(self):

        
        super().__init__()
        pass

    def forward(self, x):
        pass

class Image2BiomassModel(nn.Module):

    def __init__(self):
        super().__init__()

        # ---- load DINOv2 giant backbone from local ----
        self.backbone = Dinov2Model.from_pretrained(
            # "facebook/dinov2-giant"
            "/mnt/d/Sayid/Projects/Image2Biomass/CSIRO-Image2Biomass-Prediction/dinov2"
        )

        # self.backbone = BackBone()
        # backbone = resnet152(weights=ResNet152_Weights.IMAGENET1K_V2)
        # backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        # self.backbone = nn.Sequential(*list(backbone.children())[:-1])

        for param in self.backbone.parameters():
            param.requires_grad = False
        

        self.noise = nn.Sequential(
            nn.AlphaDropout(0.1),
        )
        # DINOv2-giant outputs 1536-dim features
        self.fc1 = nn.Sequential(
            # nn.Linear(2048, 1024),
            nn.Linear(1536, 1024),
            nn.BatchNorm1d(1024),
            nn.Mish(),
            nn.Dropout(0.4),
        )
        # self.fc1 = nn.Sequential(
        #     nn.Linear(1536, 1024),
        #     nn.BatchNorm1d(1024),
        #     nn.Mish(),
        #     nn.Dropout(0.4),
        # )

        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.Mish(),
            nn.Dropout(0.4),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.Mish(),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
        )

        self.residual = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.Mish(),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
        )

        self.out = nn.Linear(512, 3)

        self.criterion = nn.SmoothL1Loss(beta=0.5)

    def forward(self, x, y=None):
        # DINOv2-giant expects normalized images and outputs [B, 1536]
        outputs = self.backbone(x)
        x = outputs.last_hidden_state[:, 0]  # Take [CLS] token
        #END OF DINOV2

        # START OF RESNET
        # x = self.backbone(x)
        # x = x.view(x.size(0), -1)
        # END OF RESNET
        x = self.noise(x)
        x = self.fc1(x)
        x = self.fc2(x)
        

        res = self.residual(x)
        x = x + res
        x = F.mish(x)

        preds = self.out(x)

        loss = None
        if y is not None:
            loss = self.criterion(preds, y)

        return preds, loss



# sample = next(iter(train_dataloader))
# model = Image2BiomassModel()

# model(sample[0], sample[2])

In [12]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Train Loop

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Image2BiomassModel().to(device)
BATCH_SIZE=8
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# base_optimizer = torch.optim.AdamW
# optimizer = SAM(model.parameters(), base_optimizer, lr=1e-4, weight_decay=1e-2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
weights = torch.tensor([0.1, 0.1, 0.1, 0.2, 0.5], device=device)

train_losses, val_losses = [], []
train_r2_history, val_r2_history = [], []

In [14]:
def weighted_r2(y_true, y_pred, weights):
    y_true = target_untransform(y_true)
    y_pred = target_untransform(y_pred)

    
    # create new columns
    gdm = (y_true[:, 0] + y_true[:, 2]).unsqueeze(1)   # (batch, 1)
    tot = (y_true[:, 0] + y_true[:, 1] + y_true[:, 2]).unsqueeze(1)
    
    gdm_pred = (y_pred[:, 0] + y_pred[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_pred = (y_pred[:, 0] + y_pred[:, 1] + y_pred[:, 2]).unsqueeze(1)

    # append columns
    y_true = torch.cat([y_true, gdm, tot], dim=1)
    y_pred = torch.cat([y_pred, gdm_pred, tot_pred], dim=1)

    # print("Prediction:", y_pred)
    # print("Target:", y_true)

    # compute weighted R2
    mean = y_true.mean(dim=0)
    SSE = ((y_true - y_pred)**2).sum(dim=0)
    TSS = ((y_true - mean)**2).sum(dim=0)
    TSS = torch.clamp(TSS, min=1e-8)
    R2 = 1 - SSE / TSS
    R2 = torch.clamp(R2, min=-10, max=1)
    return (R2 * weights).sum() / weights.sum()


def weighted_r2_single(y_true, y_pred):
    """
    Compute R2 for each individual target separately.
    Returns dict with R2 for each target:
    - Dry_Green_g (y[0])
    - Dry_Dead_g (y[1])
    - Dry_Clover_g (y[2])
    - GDM_g (y[0] + y[2])
    - Dry_Total_g (y[0] + y[1] + y[2])
    """
    y_true = target_untransform(y_true)
    y_pred = target_untransform(y_pred)
    
    # create new columns for GDM and Total
    gdm_true = (y_true[:, 0] + y_true[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_true = (y_true[:, 0] + y_true[:, 1] + y_true[:, 2]).unsqueeze(1)
    
    gdm_pred = (y_pred[:, 0] + y_pred[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_pred = (y_pred[:, 0] + y_pred[:, 1] + y_pred[:, 2]).unsqueeze(1)
    
    # append columns
    y_true_full = torch.cat([y_true, gdm_true, tot_true], dim=1)
    y_pred_full = torch.cat([y_pred, gdm_pred, tot_pred], dim=1)
    
    # compute R2 for each target separately
    mean = y_true_full.mean(dim=0)  # (5,)
    SSE = ((y_true_full - y_pred_full)**2).sum(dim=0)  # (5,)
    TSS = ((y_true_full - mean)**2).sum(dim=0)  # (5,)
    TSS = torch.clamp(TSS, min=1e-8)
    R2 = 1 - SSE / TSS  # (5,)
    R2 = torch.clamp(R2, min=-10, max=1)
    
    target_labels = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    
    return {label: r2_val.item() for label, r2_val in zip(target_labels, R2)}

In [15]:
%%capture
!pip install wandb

In [16]:
import wandb
import os
os.environ["WANDB_API_KEY"] = "f5498d8776689da0795dbdee5044ad07e5c956ad"
wandb.login(key=os.environ["WANDB_API_KEY"])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ser/.netrc
wandb: Currently logged in as: sayid-10121012 (sayid-10121012-universitas-komputer-indonesia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [17]:
import wandb

#HYPERPARAMETERS
run = wandb.init(
    project="IMAGE2BIOMASSPREDICTION",
    config={
        "architecture": "dinov2-giant-frozen",
        "dataset": "Image2Biomass",
        "epochs": 1000,
    },
)

wandb.watch(model, log="all", log_freq=100)

In [ ]:
from tqdm import tqdm
import torch
from torch.nn.utils import clip_grad_norm_

epochs = 1000
best_val_r2 = -float('inf')  # Track best validation R2
best_epoch = 0

for epoch in range(1, epochs+1):
    model.train()
    train_loss = 0
    train_r2_scores = []
    train_r2_individual = {
        "Dry_Green_g": [],
        "Dry_Dead_g": [],
        "Dry_Clover_g": [],
        "GDM_g": [],
        "Dry_Total_g": []
    }

    for imgs, _, y in tqdm(train_dataloader, desc=f"[Train] Epoch {epoch}"):

        imgs, y = imgs.to(device), y.to(device)

        # preds, loss = model(imgs, y)
        # optimizer.zero_grad()
        # print(loss.requires_grad)
        # def closure():
        #     # optimizer.zero_grad()
        #     loss.backward()
        #     return loss
        # # loss.backward()
        # optimizer.step(closure)
        
        preds, loss = model(imgs, y)

        # L1 REGULARIZATION
        l1_lambda = 1e-7
        reg_loss = sum(param.abs().sum() for param in model.parameters())
        loss = loss + l1_lambda * reg_loss
        optimizer.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        train_r2_scores.append(weighted_r2(y, preds, weights).item())
        
        # Track individual R2 scores
        r2_dict = weighted_r2_single(y, preds)
        for target_name, r2_value in r2_dict.items():
            train_r2_individual[target_name].append(r2_value)

    avg_train_loss = train_loss / len(train_dataloader)
    avg_train_r2 = sum(train_r2_scores) / len(train_r2_scores)
    avg_train_r2_individual = {k: sum(v) / len(v) for k, v in train_r2_individual.items()}

    # VALIDATION
    model.eval()
    val_loss = 0
    val_r2_scores = []
    val_r2_individual = {
        "Dry_Green_g": [],
        "Dry_Dead_g": [],
        "Dry_Clover_g": [],
        "GDM_g": [],
        "Dry_Total_g": []
    }

    with torch.no_grad():
        for imgs, _, y in tqdm(val_dataloader, desc=f"[Val] Epoch {epoch}"):
            imgs, y = imgs.to(device), y.to(device)
            preds, loss = model(imgs, y)
            val_loss += loss.item()
            val_r2_scores.append(weighted_r2(y, preds, weights).item())
            
            # Track individual R2 scores
            r2_dict = weighted_r2_single(y, preds)
            for target_name, r2_value in r2_dict.items():
                val_r2_individual[target_name].append(r2_value)

    avg_val_loss = val_loss / len(val_dataloader)
    avg_val_r2 = sum(val_r2_scores) / len(val_r2_scores)
    avg_val_r2_individual = {k: sum(v) / len(v) for k, v in val_r2_individual.items()}
    
    val_losses.append(avg_val_loss)
    train_losses.append(avg_train_loss)
    val_r2_history.append(avg_val_r2)
    train_r2_history.append(avg_train_r2)
    
    # Save best model based on validation R2
    if avg_val_r2 > best_val_r2:
        best_val_r2 = avg_val_r2
        best_epoch = epoch
        torch.save(model.state_dict(), "image2biomass_weights_resnet50.pth")
        print(f"✓ New best model saved! Val R2: {best_val_r2:.4f} at epoch {epoch}")
    
    # Log to wandb with individual R2 scores
    wandb.log({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "train_r2": avg_train_r2,
        "train_r2_Dry_Green_g": avg_train_r2_individual["Dry_Green_g"],
        "train_r2_Dry_Dead_g": avg_train_r2_individual["Dry_Dead_g"],
        "train_r2_Dry_Clover_g": avg_train_r2_individual["Dry_Clover_g"],
        "train_r2_GDM_g": avg_train_r2_individual["GDM_g"],
        "train_r2_Dry_Total_g": avg_train_r2_individual["Dry_Total_g"],
        "val_loss": avg_val_loss,
        "val_r2": avg_val_r2,
        "val_r2_Dry_Green_g": avg_val_r2_individual["Dry_Green_g"],
        "val_r2_Dry_Dead_g": avg_val_r2_individual["Dry_Dead_g"],
        "val_r2_Dry_Clover_g": avg_val_r2_individual["Dry_Clover_g"],
        "val_r2_GDM_g": avg_val_r2_individual["GDM_g"],
        "val_r2_Dry_Total_g": avg_val_r2_individual["Dry_Total_g"],
        "best_val_r2": best_val_r2,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | "
          f"Train R2: {avg_train_r2:.4f} | Val Loss: {avg_val_loss:.4f} | Val R2: {avg_val_r2:.4f}")
    print(f"  Train R2 by target: Green={avg_train_r2_individual['Dry_Green_g']:.4f}, "
          f"Dead={avg_train_r2_individual['Dry_Dead_g']:.4f}, "
          f"Clover={avg_train_r2_individual['Dry_Clover_g']:.4f}, "
          f"GDM={avg_train_r2_individual['GDM_g']:.4f}, "
          f"Total={avg_train_r2_individual['Dry_Total_g']:.4f}")
    print(f"  Val R2 by target: Green={avg_val_r2_individual['Dry_Green_g']:.4f}, "
          f"Dead={avg_val_r2_individual['Dry_Dead_g']:.4f}, "
          f"Clover={avg_val_r2_individual['Dry_Clover_g']:.4f}, "
          f"GDM={avg_val_r2_individual['GDM_g']:.4f}, "
          f"Total={avg_val_r2_individual['Dry_Total_g']:.4f}")

print(f"\n{'='*60}")
print(f"Training completed!")
print(f"Best validation R2: {best_val_r2:.4f} achieved at epoch {best_epoch}")
print(f"Best model saved to: image2biomass_weights_resnet50.pth")
print(f"{'='*60}")

[Val] Epoch 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:11<00:00,  1.28s/it]


✓ New best model saved! Val R2: -2.9182 at epoch 1
Epoch 1 | Train Loss: 1.9119 | Train R2: -0.9632 | Val Loss: 0.6065 | Val R2: -2.9182
  Train R2 by target: Green=-1.0686, Dead=-0.4814, Clover=-0.5542, GDM=-1.0835, Total=-1.0722
  Val R2 by target: Green=-1.2710, Dead=-3.6674, Clover=-3.6543, GDM=-2.5226, Total=-3.1087


[Val] Epoch 2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:10<00:00,  1.12s/it]


✓ New best model saved! Val R2: -2.1778 at epoch 2
Epoch 2 | Train Loss: 1.7238 | Train R2: -0.5366 | Val Loss: 0.5675 | Val R2: -2.1778
  Train R2 by target: Green=-0.3727, Dead=-0.4948, Clover=-0.8432, GDM=-0.7749, Total=-0.4210
  Val R2 by target: Green=-1.2899, Dead=-0.2457, Clover=-3.8406, GDM=-2.8134, Total=-2.1550


[Val] Epoch 3: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:10<00:00,  1.12s/it]


...

[Val] Epoch 5: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:10<00:00,  1.12s/it]


✓ New best model saved! Val R2: -0.6290 at epoch 5
Epoch 5 | Train Loss: 1.6349 | Train R2: -0.1545 | Val Loss: 0.5140 | Val R2: -0.6290
  Train R2 by target: Green=0.0910, Dead=-0.3399, Clover=-0.4297, GDM=-0.1476, Total=-0.1143
  Val R2 by target: Green=0.1593, Dead=-1.8743, Clover=-0.0040, GDM=-0.3528, Total=-0.7730


[Val] Epoch 6: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:09<00:00,  1.10s/it]


Epoch 6 | Train Loss: 1.5977 | Train R2: -0.0397 | Val Loss: 0.5329 | Val R2: -3.6581
  Train R2 by target: Green=0.0475, Dead=-0.1400, Clover=-0.1179, GDM=-0.0901, Total=-0.0014
  Val R2 by target: Green=-1.6367, Dead=0.0055, Clover=-6.7358, GDM=-4.5426, Total=-3.8258


[Val] Epoch 7:  11%|██████████▌                                                                                    | 1/9 [00:01<00:08,  1.10s/it]

In [ ]:
wandb.finish()

In [ ]:
torch.save(model.state_dict(), "image2biomass_weights_submission.pth")
print("Model saved to image2biomass_weights_submission.pth")

In [ ]:
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

model = Image2BiomassModel().to(device)
model.load_state_dict(torch.load("image2biomass_weights_resnet50.pth", map_location=device))
model.eval()

rows = []

target_cols = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "GDM_g",
    "Dry_Total_g",
]

test_dataset = Image2BioMassTestDataset(
    dataset_path="/kaggle/input/csiro-biomass/",
    img_transform=val_transform
    # img_transform=image_transform,
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

with torch.no_grad():
    for imgs, _, sample_ids in tqdm(test_dataloader, desc="Inference"):
        imgs = imgs.to(device)

        # (B, 3) - model outputs: [Dry_Green_g, Dry_Dead_g, Dry_Clover_g]
        y_pred, _ = model(imgs, y=None)
        print("y_pred transformed:", y_pred)

        y_pred = target_untransform(y_pred).cpu().numpy()
        
        print("y_pred pure:", y_pred)
        # extract 3 predictions in the correct order
        dg = y_pred[:, 0]  # Dry_Green_g
        dd = y_pred[:, 1]  # Dry_Dead_g
        dc = y_pred[:, 2]  # Dry_Clover_g

        # compute extra targets
        gdm = dg + dc
        dry_total = dg + dd + dc

        preds5 = np.stack([dg, dd, dc, gdm, dry_total], axis=1)
        np.set_printoptions(suppress=True, precision=4)
        # print(preds5)

        # build submission rows
        for sid, pred_vec in zip(sample_ids, preds5):
            for col, value in zip(target_cols, pred_vec):
                rows.append({
                    "sample_id": f"{sid}__{col}",
                    "target": float(value)
                })

df_submit = pd.DataFrame(rows)
df_submit.to_csv("submission.csv", index=False)
print("Saved submission.csv")
df_submit.head(20)

In [ ]:
import shutil

shutil.make_archive("model_weights", "zip", "/kaggle/working", "image2biomass_weights_resnet50.pth")
from IPython.display import FileLink
FileLink("model_weights.zip")

In [ ]:
# target_untransform(3.8318)